# Small-Scale AutoSchemaKG v1 — local Qwen3.5-2B

This notebook runs the first end-to-end version with a local Qwen model. Select a GPU runtime, then run all cells in order. No commercial LLM API key is required.

In [ ]:
import os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
REPO_URL = 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git'
REPO_DIR = '/content/SmallScaledAutoSchemaKG'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Repository:', os.getcwd())

## Install
Qwen3.5 support follows recent vLLM releases, so this cell uses the vLLM nightly wheel index. Installation can take several minutes.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--system', 'vllm', '--torch-backend=auto',
    '--extra-index-url', 'https://wheels.vllm.ai/nightly'
], check=True)
# vLLM may replace Colab's PyTorch build. Remove the preinstalled CUDA
# multimedia wheels because this text-only workflow does not use them and
# stale TorchAudio/TorchVision wheels can target a different CUDA version.
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchvision'
], check=False)
subprocess.run([
    sys.executable, '-c',
    "import torch, vllm; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'vLLM', vllm.__version__)"
], check=True)

## Start the local model server
The 2B checkpoint is the closest smaller official Qwen3.5 model to the requested 3B scale. Change `MODEL_ID` to `Qwen/Qwen3.5-4B` only if the assigned GPU has enough memory.

In [ ]:
import shutil, time
import requests

MODEL_ID = 'Qwen/Qwen3.5-2B'
PORT = 8000
LOG_PATH = '/content/qwen35_vllm.log'
vllm_executable = shutil.which('vllm')
if not vllm_executable:
    raise RuntimeError('vLLM executable was not installed')
server_log = open(LOG_PATH, 'w', encoding='utf-8')
server_cmd = [
    vllm_executable, 'serve', MODEL_ID,
    '--host', '127.0.0.1', '--port', str(PORT),
    '--dtype', 'half', '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.85', '--language-model-only'
]
server = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError(f'vLLM stopped early. Read {LOG_PATH}')
    try:
        response = requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5)
        if response.ok:
            print('Local model server is ready:', response.json())
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f'vLLM did not become ready. Read {LOG_PATH}')

## Run extraction, schema induction, and GraphML export

In [ ]:
OUTPUT_DIR = '/content/colab_outputs/v1'
subprocess.run([
    sys.executable, 'scripts/run_colab_v1.py',
    '--model', MODEL_ID,
    '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--output-dir', OUTPUT_DIR,
    '--overwrite'
], check=True)

In [ ]:
import json
from pathlib import Path
summary = json.loads(Path(OUTPUT_DIR, 'run_summary.json').read_text())
print(json.dumps(summary, indent=2))
assert summary.get('nodes', 0) > 0, 'The graph contains no nodes'
assert summary.get('edges', 0) > 0, 'The graph contains no edges'

## Package the outputs
Download the resulting zip from the Colab file browser.

In [ ]:
archive = shutil.make_archive('/content/autoschemakg_colab_v1', 'zip', OUTPUT_DIR)
print('Created:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass